In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# step 1
class FinderModel(nn.Module):
    def __init__(self, img_size=250):
        super().__init__()

        # find simple patterns (edges, corners)
        self.conv1 = nn.Conv2d(in_channels=3,out_channels=16,kernel_size=3,padding=1)

        self.rel1=nn.ReLU()

        self.poll1=nn.MaxPool2d(kernel_size=2)

        # find bigger patterns (eyes, ears)
        self.conv2=nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,padding=1)

        self.rel2=nn.ReLU()

        self.poll2=nn.MaxPool2d(kernel_size=2)

        # find whole shapes (faces, bodies)
        self.conv3=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1)

        self.rel3=nn.ReLU()

        self.poll3=nn.MaxPool2d(kernel_size=2)
        self.flater=nn.Flatten()

        self.drop=nn.Dropout(0.5)

        # each pool halves the picture, so 3 pools shrink it 8x (250 -> 125 -> 62 -> 31)
        side = img_size // 8
        self.final=nn.Linear(in_features=64*side*side,out_features=2)


    def forward(self, x):
        out=self.conv1(x)
        out =self.rel1(out)
        out =self.poll1(out)
        out =self.conv2(out)
        out =self.rel2(out)
        out =self.poll2(out)
        out =self.conv3(out)
        out =self.rel3(out)
        out =self.poll3(out)
        out =self.flater(out)
        out =self.drop(out)
        out=self.final(out)
        return out


In [3]:
#step 2
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"using device-{device}")


# picture size. drop this to 128 for roughly 4x faster epochs (the model adjusts itself)
IMG_SIZE = 250

# training photos: same size, plus small random changes so the model cannot memorise them
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),      # mirror image: a flipped cat is still a cat
    transforms.RandomRotation(10),          # tilt up to 10 degrees
    transforms.ColorJitter(0.2, 0.2),       # a bit brighter or darker
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# exam photos: same size, but no random changes - grading has to be fair
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_data_path="archive/training_set/training_set"
test_data_path="archive/test_set/test_set"

train_dataset=datasets.ImageFolder(root=train_data_path,transform=train_transform)
test_dataset=datasets.ImageFolder(root=test_data_path,transform=test_transform)

# shuffle the study photos, but not the exam ones (order does not matter when grading)
train_dataloader=DataLoader(dataset=train_dataset,batch_size=32,shuffle=True,num_workers=4)
test_dataloader=DataLoader(dataset=test_dataset,batch_size=32,shuffle=False,num_workers=4)

print(f"{len(train_dataset)} training photos, {len(test_dataset)} test photos")


using device-mps
8005 training photos, 2023 test photos


In [4]:
#step 3

epochs=20
model=FinderModel(img_size=IMG_SIZE).to(device=device)

loss_fun=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)


def check_score():
    """Grade the model on the test photos, give back the % it got right."""
    model.eval()                                       # exam mode: dropout switched off
    correct = 0
    total = 0
    with torch.no_grad():                              # just guessing, no learning
        for images, labels in test_dataloader:
            images, labels = images.to(device), labels.to(device)
            guess = model(images)
            _, final = torch.max(guess, dim=1)         # take the higher of the 2 scores
            correct += (final == labels).sum().item()
            total += labels.size(0)
    model.train()                                      # back to learning mode
    return correct / total * 100


best_score = 0

model.train()
for epoch in range(epochs):
    total_loss = 0
    for images, labels in train_dataloader:

        images,labels=images.to(device) , labels.to(device)

        optimizer.zero_grad()
        guess =model(images)

        loss =loss_fun(guess,labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    # grade after every epoch, so you can see when it stops improving
    score = check_score()
    print(f"Epoch {epoch+1}: avg loss {total_loss / len(train_dataloader):.4f}  test {score:.1f}%")

    # keep a copy of the best model so far, so a bad later epoch cannot spoil it
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), "cat_dog.pth")

print(f"best test score- {best_score:.1f}% (saved to cat_dog.pth)")


Epoch 1: avg loss 0.6646  test 64.6%
Epoch 2: avg loss 0.5879  test 70.4%
Epoch 3: avg loss 0.5482  test 69.4%
Epoch 4: avg loss 0.5247  test 76.5%
Epoch 5: avg loss 0.5081  test 73.0%
Epoch 6: avg loss 0.4877  test 78.9%
Epoch 7: avg loss 0.4749  test 79.0%
Epoch 8: avg loss 0.4620  test 80.8%
Epoch 9: avg loss 0.4414  test 79.6%
Epoch 10: avg loss 0.4417  test 81.5%
Epoch 11: avg loss 0.4238  test 80.2%
Epoch 12: avg loss 0.4110  test 78.1%
Epoch 13: avg loss 0.4146  test 80.8%
Epoch 14: avg loss 0.3983  test 81.8%
Epoch 15: avg loss 0.3880  test 79.4%
Epoch 16: avg loss 0.3786  test 82.0%
Epoch 17: avg loss 0.3857  test 84.2%
Epoch 18: avg loss 0.3748  test 83.8%
Epoch 19: avg loss 0.3625  test 84.6%
Epoch 20: avg loss 0.3691  test 84.8%
best test score- 84.8% (saved to cat_dog.pth)


In [5]:
#step 4

# bring back the best epoch's model (the one saved during training) and grade it
model.load_state_dict(torch.load("cat_dog.pth", map_location=device))

print(f"final test score- {check_score():.1f}%")


final test score- 84.8%
